In [2]:
from infostop import Infostop
import numpy as np
import pandas as pd
import polars as pl
import geopandas as gpd
import matplotlib.pyplot as plt

import yaml

In [6]:
with open("05_detect_stops_config.yaml") as fp:
    config = yaml.safe_load(fp)


In [ ]:
model = Infostop()
labels = model.fit_predict(data)

In [ ]:
def initial_filter(df: pl.DataFrame, min_pings: int) -> pl.DataFrame:
    """Keep only devices with at least min_pings pings."""
    valid_devices = (
        df.group_by("device_id")
        .len()
        .filter(pl.col("len") >= min_pings)["device_id"]
        .to_list()
    )
    return df.filter(pl.col("device_id").is_in(valid_devices))


In [ ]:
data = pl.read_parquet(
    "/mnt/common-hdd/group-dirs/tkom-2026-project/partitioned/00/pings.parquet"
)


In [ ]:
filtered = initial_filter(data, min_pings=config["filtering"]["min_pings_per_day"])

In [ ]:
filtered = filtered.with_columns(
    pl.col("timestamp").dt.epoch(time_unit="s").alias("epoch")
)

In [7]:
# stop detection process
params = config["infostop"]

model = Infostop(
    r1=params["r1"],
    r2=params["r2"],
    min_staying_time=params["min_staying_time"],
    max_time_between=params["max_time_between"],
    min_size=params["min_size"],
    distance_metric=params["distance_metric"],
)

In [17]:
# run infostop per device — data already sorted, no re-sort needed in loop
# catch devices with no stop events (all pings labelled -1)
results = []
for device_id, group in filtered.to_pandas().groupby("device_id", sort=False):
    coords = group[["latitude", "longitude", "epoch"]].to_numpy()
    try:
        labels = model.fit_predict(coords)
    except Exception:
        labels = np.full(len(coords), -1)
    group = group.copy()
    group["stop_label"] = labels
    results.append(group)

stops_df = pd.concat(results, ignore_index=True)

n_stopped = (stops_df["stop_label"] >= 0).sum()
print(
    f"Stop pings:    {n_stopped:,}  ({n_stopped / len(stops_df):.1%} of filtered pings)"
)
print(f"Unique stops:  {stops_df['stop_label'].clip(lower=0).nunique():,}")
stops_df.head(3)


Stop pings:    9,474,653  (61.3% of filtered pings)
Unique stops:  393


,timestamp,device_id,device_aid_type,latitude,longitude,epoch,stop_label
0,2024-10-02 02:37:09+02:00,1,AAID,47.479920,19.074155,1727829429,2
1,2024-10-02 03:17:49+02:00,1,AAID,47.479995,19.074221,1727831869,2
2,2024-10-02 03:26:50+02:00,1,AAID,47.476752,19.075154,1727832410,-1


In [21]:
stops = pl.from_pandas(stops_df).drop("epoch")

In [ ]:
stops.write_parquet("/mnt/common-hdd/group-dirs/tkom-2026-project/stops/00.parquet")